> **Chapter 15, Part 6** | Engineering lens. **Focus:** every concept we built has a one-to-one Dagster primitive. This notebook is the Rosetta stone. The Dagster code is illustration, not executed.

# From Our Toy to Dagster

We built an orchestrator in about 150 lines. Dagster is that idea, hardened for production: a web UI, a database of run history, a daemon for schedules and sensors, retries, partitions, freshness, and integrations. The concepts map one to one. This notebook is the translation table.

**The Dagster code below is illustrative and is not executed in this notebook.** Running it needs `pip install dagster dagster-webserver` and a `dagster dev` process. The point is to recognize each primitive as something you already built.

## Assets

Our toy:

```python
g.add(Asset("stg_events", ["raw_events"], compute_stg))
```

Dagster:

```python
from dagster import asset

@asset
def raw_events() -> list:
    return list(range(100))

@asset
def stg_events(raw_events: list) -> list:        # the parameter name IS the dependency
    return [x for x in raw_events if x % 2 == 0]
```

Dagster reads the dependency from the function signature. Our `deps=["raw_events"]` is the same edge, written by hand. The `@asset` decorator registers the function the way `g.add(...)` registered ours.

## Partitions and backfills

Our toy:

```python
backfill(g, "rollup", week, log)          # log skips what is already done
```

Dagster:

```python
from dagster import asset, DailyPartitionsDefinition

daily = DailyPartitionsDefinition(start_date="2026-06-01")

@asset(partitions_def=daily)
def rollup(context):
    day = context.partition_key
    ...
```

Dagster's backfills are launched from the UI or `dagster job backfill`, and its storage is the materialization log we hand-built. Re-running a materialized partition is a no-op for the same reason ours was: the system already recorded it.

## Sensors and freshness

Our toy:

```python
Sensor("new_file", file_waiting, process_file).poll(clock)
FreshnessPolicy(max_age=4).is_stale(last, now)
```

Dagster:

```python
from dagster import sensor, RunRequest, FreshnessPolicy, asset

@sensor(target=rollup)
def new_file_sensor(context):
    if file_has_landed():
        yield RunRequest(partition_key=today())

@asset(legacy_freshness_policy=FreshnessPolicy(maximum_lag_minutes=240))
def rollup(): ...
```

Same two ideas: a polled condition that yields work, and a max-age that marks an asset stale. Dagster's daemon runs the poll loop we ran by hand with `for _ in range(10)`.

## Retries and the run graph

Our toy:

```python
materialize_with_failures(g, retry=RetryPolicy(max_attempts=3))
```

Dagster:

```python
from dagster import asset, RetryPolicy, Backoff

@asset(retry_policy=RetryPolicy(max_retries=3, delay=2, backoff=Backoff.EXPONENTIAL))
def clean_a(): ...
```

Skip-on-failure is automatic in Dagster: if an upstream asset fails, downstream assets in the same run are not materialized, exactly as our `status == "skipped"` branch enforced.

## Wiring it together

Dagster collects assets, schedules, and sensors into a `Definitions` object, which is the deployable unit:

```python
from dagster import Definitions

defs = Definitions(
    assets=[raw_events, stg_events, rollup, clean_a],
    sensors=[new_file_sensor],
)
```

That object is our `AssetGraph` plus the reactive pieces, registered for a long-running daemon instead of a single `materialize()` call.

## Why Dagster here, and not Airflow or Prefect

Airflow is task-first: you author a DAG of operators, and data assets are implicit. It is the incumbent, with the largest operator ecosystem, and it is the right answer when you have hundreds of existing Airflow DAGs. Prefect is Python-function-first with a light touch and excellent dynamic-workflow ergonomics, strong when your control flow is irregular. Dagster is asset-first, which is why this chapter uses it: the asset graph is the same object as the lineage graph from Chapter 12, so the mental model carries straight through. None of the three is wrong. The asset framing is simply the one that made this chapter's pure-Python core the shortest path to the idea.

Next: put the whole apparatus to work on the repo's own trading platform.